# 第3日 ノートブック01：YOLO を理解する

# Day 3 Notebook 01: Understanding YOLO

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 物体検出モデルが何をするのか説明できる | 1. Explain what an object detection model does |
| 2. バウンディングボックスと座標の表し方を説明できる | 2. Explain bounding boxes and their coordinates |
| 3. 確からしさ（confidence）の意味を説明できる | 3. Explain what confidence means |
| 4. IoU を手計算できる | 4. Compute IoU by hand |
| 5. 事前学習モデルが PVC 欠陥を検出できない理由を説明できる | 5. Explain why a pretrained model cannot detect PVC defects |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 80 分 / 80 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第1日・第2日を終えていること。 / You have finished Days 1 and 2. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| IoU の手計算結果 | Your hand-computed IoU |
| YOLO 形式の座標計算 | A YOLO-format coordinate calculation |

### confidence と正しさは別 / confidence is not correctness

![](../../assets/diagrams/teach_score_vs_correctness.png)

図：5 つの検出の confidence と、正解と比べた正しさ（値は例示用）。

Figure: confidence against correctness for five detections (illustrative values).

**気づいてほしいこと**

confidence が高い検出でも間違っていることがあり、低い検出でも正しいことがあります。confidence はモデルが出したスコアで、正解でも「正しい確率」でもありません。正しさは正解ラベルと対応づけのルールで確かめます。

**What to notice**

A high-confidence detection can be wrong and a low-confidence one can be right. Confidence is a score produced by the model: it is neither ground truth nor a probability of being correct. Correctness is checked against ground truth with a stated matching rule.

**考えてみてください / Try to answer**

- しきい値 0.60 で残る検出のうち、正しくないのはどれですか。
- At a threshold of 0.60, which of the surviving detections are wrong?


### 予想して、実行して、説明する / Predict, run, inspect, explain

クラスは欠陥の種類です。confidence はモデルのスコアで、正解の保証ではありません。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。x2 だけを変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

A class is a defect category; confidence is a model score, not guaranteed correctness. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only x2, predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.


## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第3日 / Day 3 |
| この日の順番 / Position today | 1 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 80 分 / 80 minutes |
| 前提 / Prerequisites | 第2日 / Day 2 |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第3日 / Day 3](../../COURSE_SYLLABUS.md#day3) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

#### YOLO とは
YOLO は **物体検出** のための AI モデルです。名前は
**You Only Look Once**（一度だけ見る）の頭文字です。

従来の方法では、画像を何度も走査して「ここに物体があるか？」を繰り返し調べました。
YOLO は **1 回の計算** で、画像全体の物体の位置と種類をまとめて予測します。
そのため **高速** で、製造ラインのような連続処理に向いています。

```text
入力：画像 1 枚
出力：複数の「四角 + 種類 + 確からしさ」
```

重要なのは、YOLO は「何かある」とは言うが、
**「何があるか」は学習した種類の中からしか答えられない** という点です。

#### What YOLO is
YOLO is an AI model for **object detection**. The name means
**You Only Look Once**.

Older methods scanned an image repeatedly asking "is there an object here?". YOLO predicts all
objects in the image in a **single pass**, which makes it **fast** and therefore suitable for
continuous production lines.

```text
input : one image
output: several "box + class + confidence" results
```

Crucially, YOLO can only answer **with the classes it was trained on**.

---

## Part 1　バウンディングボックス / Bounding boxes

### 四角を画像の上で読む / reading a box on the image

![](../../assets/diagrams/teach_bbox_coords.png)

図：同じ 1 つの欠陥の四角を、画素の数（左）と画像サイズで割った値（右）で表したもの。座標は計算して図に描いています。

Figure: one defect box written as pixel counts (left) and as fractions of the image size (right). The coordinates are computed and drawn, not sketched.

**気づいてほしいこと**

左は画素の xyxy、右は画像サイズで割った cx, cy, w, h で、**同じ箱**です。画像全体の縮小・拡大では 0〜1 の値は意味を保ちますが、切り抜き（crop）や余白の追加（letterbox）では座標が変わります。API ごとに形式が違うので名前を確認します（Ultralytics の `xyxy`、OpenCV の `boundingRect` は xywh）。

**What to notice**

The left is pixel xyxy; the right is cx, cy, w, h divided by the image size — **the same box**. Resizing the whole image keeps the 0–1 values meaningful, but cropping or letterbox padding changes the coordinates. Formats differ by API (Ultralytics `xyxy`; OpenCV `boundingRect` is xywh).

**考えてみてください / Try to answer**

- 画像の左半分だけを切り抜いたとき、もとの cx = 0.6 の箱はどうなりますか。
- If you crop the left half of the image, what happens to a box with cx = 0.6?


#### 四角で囲む、4 つの数値
バウンディングボックス（以下「ボックス」）は、欠陥を囲む四角です。
ボックスは 4 つの数値で表し、並べ方（**座標の形式**）は API ごとに決まっています。

| 形式 | 4 つの数値 | 使う場所 |
|---|---|---|
| `xyxy` | `x1, y1, x2, y2`（左上と右下） | Ultralytics の `result.boxes.xyxy`、描画ライブラリ |
| `xywh` | `x, y, w, h`（左上と幅・高さ） | `cv2.boundingRect()`（OpenCV の輪郭の外接矩形） |
| `xc, yc, w, h`（正規化） | 中心と幅・高さを画像サイズで割った 0〜1 の値 | YOLO のラベルファイル（正解データ） |

同じ「4 つの数値」でも、`xyxy`・`xywh`・YOLO のラベルは **意味も順番も違います**。
関数を呼ぶたびに、どの形式が返るのかを確認してください。

YOLO のラベルは **中心方式** で、しかも **画像の幅・高さで割った 0〜1 の値**（正規化）です。
この正規化座標は、**画像全体を同じ比率で縮小・拡大する**（全体をリサイズする）ときは同じ値のまま使えます。
一方、画像の一部を切り抜いた（crop）ときや、縦横比を保つために余白を足した（letterbox）ときは、
座標の基準になる枠が変わるので、値はそのままでは使えません。
基準の枠が変わったら、新しい枠の幅・高さで計算し直します。

#### A rectangle is four numbers
A bounding box marks the defect with a rectangle. It is written as four numbers, and the
**order** of those numbers (**the coordinate format**) is fixed by each API.

| Format | Four numbers | Where it is used |
|---|---|---|
| `xyxy` | `x1, y1, x2, y2` (top-left, bottom-right) | Ultralytics `result.boxes.xyxy`, drawing libraries |
| `xywh` | `x, y, w, h` (top-left, width, height) | `cv2.boundingRect()` (the OpenCV contour rectangle) |
| `xc, yc, w, h` (normalised) | centre and size divided by the image size, in 0-1 | YOLO label files (ground truth) |

The same four numbers mean different things in `xyxy`, in `xywh` and in a YOLO label. Check which
format each function returns.

YOLO labels use the **centre style**, normalised **by the image width and height** into 0-1.
These normalised values stay valid when the **whole image** is resized by the same factor.
They do not stay valid after a crop, or after letterbox padding added to keep the aspect ratio,
because the reference frame changes. Recompute the values against the new frame.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_annotation.png")],
     ["YOLO annotation: normalised centre, width, height"], figsize=(9, 7))

#### 座標変換を自分で計算する
ピクセル座標 `(x1, y1, x2, y2)` から YOLO 形式への変換は 4 行です。

```text
xc = (x1 + x2) / 2 / 画像の幅
yc = (y1 + y2) / 2 / 画像の高さ
w  = (x2 - x1)     / 画像の幅
h  = (y2 - y1)     / 画像の高さ
```

割るのを忘れると、値が 1 を超えます。**0〜1 の範囲かどうかを必ず確認** してください。

この 4 行が使うのは **元の画像全体の幅と高さ** です。切り抜き（crop）や letterbox の余白を
入れたあとなら、新しい画像の幅・高さで計算し直してください。古い値をそのまま使うと枠がずれます。

#### Do the conversion yourself
Converting pixel corners to YOLO format takes four lines:

```text
xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w  = (x2 - x1)     / image_width
h  = (y2 - y1)     / image_height
```

Forgetting the division gives values above 1. **Always check the 0-1 range.**

These four lines divide by the width and height of the **whole original image**. After a crop, or
after letterbox padding, recompute with the new frame size; reusing the old values shifts the box.


### コードの読み方 / Reading the code

![](../../assets/svg/teach_bbox_one_example.svg)

図：class_id は整数で、正規化しません。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: class_id is an integer and is not normalised. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

class_id は整数で、正規化しません。 入力と変換後の値を指でたどってください。

**What to notice**

class_id is an integer and is not normalised. Trace the input and transformed values.

**考えてみてください / Try to answer**

x2 だけを24 px増やすと中心と幅はどう変わりますか。

How do centre and width change if only x2 grows by 24 px?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
image_width, image_height = 240, 240
x1, y1, x2, y2 = 96, 72, 168, 132

xc = (x1 + x2) / 2 / image_width
yc = (y1 + y2) / 2 / image_height
w = (x2 - x1) / image_width
h = (y2 - y1) / image_height

print(f"center x : {xc:.3f}")
print(f"center y : {yc:.3f}")
print(f"width    : {w:.3f}")
print(f"height   : {h:.3f}")
print()
print("0〜1 の範囲か / inside 0..1 ?", all(0 <= v <= 1 for v in (xc, yc, w, h)))
print("ラベルファイルの 1 行 / one label line:")
print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

### 実行結果の例 / Expected output

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

0〜1 の範囲か / inside 0..1 ? True
ラベルファイルの 1 行 / one label line:
0 0.550000 0.425000 0.300000 0.250000
```

This example produces the following result (the same input gives the same result).

```text
center x : 0.550
center y : 0.425
width    : 0.300
height   : 0.250

0〜1 の範囲か / inside 0..1 ? True
ラベルファイルの 1 行 / one label line:
0 0.550000 0.425000 0.300000 0.250000
```


---

## Part 2　クラスと確からしさ / Classes and confidence

#### クラスラベル
**クラス** は「種類」のことです。YOLO は、学習した種類のどれかに分類します。

一般的な事前学習モデル（COCO データセット）は 80 種類を学習しています。
例：`person`, `car`, `bus`, `dog`, `bottle` ……

ここに **`black_spot`（黒点）や `scratch`（傷）はありません**。
だから PVC の欠陥は検出できません。

#### Class label
A **class** is a category. YOLO assigns each detection to one of its trained classes.

The common pretrained model (COCO dataset) knows 80 classes, for example
`person`, `car`, `bus`, `dog`, `bottle` ...

**`black_spot` and `scratch` are not among them**, which is why it cannot detect PVC defects.

#### 確からしさ（confidence）
**confidence** は、モデルが各候補に付けた **スコア** です。0〜1 の値をとります。

| 値 | 読み方 |
|---|---|
| 0.95 | スコアが高い（モデルが強く支持している） |
| 0.50 | 中間のスコア |
| 0.10 | スコアが低い |

**このスコアは「正しい確率」ではありません。**
0.8 を「80% の確率で正しい」と読むことはできませんし、0.50 も「五分五分の確率で正しい」ではありません。
スコアは、学習したクラスの見た目にどれくらい近いかをモデルが数値にしたものです。
**スコアが高くても正しいとは限らず、低くても正しい検出はあります。**
正しさは、正解ラベルと決められた対応付けのルール（例：IoU がしきい値以上）で確認します。

#### Confidence
**Confidence** is a **score** the model assigns to each candidate, between 0 and 1.

| Value | How to read it |
|---|---|
| 0.95 | high score (the model strongly supports it) |
| 0.50 | middling score |
| 0.10 | low score |

**This score is not a probability of correctness.** You cannot read 0.8 as "80% likely to be
correct", and 0.50 does not mean an even chance of being correct. The score says how closely the
candidate matches the appearance of the class the model learned.
**A high score does not guarantee a correct detection, and a low score does not make a detection
wrong.** Correctness is checked against ground truth with a stated matching rule, for example
IoU at or above a threshold.


---

## Part 3　IoU（重なりの度合い）/ Intersection over Union

### 交差・和集合・IoU / intersection, union and IoU

![](../../assets/diagrams/teach_iou_rects.png)

図：A（正解）と B（予測）の四角を実際に重ね、交差（重なりの面積）と和集合（2 つの枠を合わせた領域。重なりは 1 回だけ数える）を計算したもの。この図では 6,750 ÷ 15,150 ≒ 0.45 です。

Figure: truth box A and predicted box B actually overlapped, with the intersection and the union (the combined area of the two boxes, with the overlap counted once) computed. Here 6,750 / 15,150 ≈ 0.45.

**気づいてほしいこと**

IoU は「重なりの度合い」を 1 つの数にしたものです。0 に近いほど重なりが少なく、1 に近いほどぴったり一致しています。**面積の比** なので、単位はありません。和集合は 2 つの面積を足すだけでは求まりません。重なりを 2 回数えないよう、1 回だけ数えます。

**What to notice**

IoU turns overlap into one number. Near 0 means little overlap; near 1 means a tight fit. It is a **ratio of areas**, so it has no unit. The union is not the sum of the two areas: the overlap is counted once, not twice.

**考えてみてください / Try to answer**

- IoU が 0.3 のとき、重なりの面積は「予測の枠」の 30 % ですか、「正解の枠」の 30 % ですか、それとも別のものの 30 % ですか。
- When IoU is 0.3, is the intersection 30 % of the predicted box, 30 % of the truth box, or 30 % of something else?


#### IoU とは「2 つの四角の重なり具合」
**IoU** は、予測したボックスと正解のボックスが
**どれくらい重なっているか** を 0〜1 で表す数値です。

```text
IoU = 重なりの面積 ÷ 2 つの枠を合わせた領域の面積（重なりは 1 回だけ数える）
```

「2 つの枠を合わせた領域」が **和集合** です。2 つの面積をそのまま足すと重なりを 2 回
数えてしまうので、重なりを 1 回だけ数えるように足し引きします。

- `IoU = 1.0` … 完全に一致
- `IoU = 0.5` … まあまあ合っている
- `IoU = 0.0` … まったく重なっていない

`IoU = 0.3` は「重なりの面積が、**2 つの枠を合わせた領域の** 30 %」という意味です。
予測の枠の 30 %、正解の枠の 30 % という意味ではありません。

一般に `IoU >= 0.5` を「正しく検出できた」とみなすことが多いです。
ただし、これは **慣習** であり、用途によって変わります。

#### IoU measures how much two boxes overlap
**IoU** expresses how much a predicted box overlaps the true box, between 0 and 1.

```text
IoU = area of the intersection / area of the union (count the overlap once)
```

The union is the combined region of the two boxes. Adding the two areas directly would count the
overlap twice, so the subtraction counts it once.

- `IoU = 1.0` identical
- `IoU = 0.5` roughly matching
- `IoU = 0.0` no overlap

`IoU = 0.3` means the intersection is 30 % of the **union of the two boxes**. It does not mean 30 %
of the predicted box, nor 30 % of the true box.

A detection is often counted as correct when `IoU >= 0.5`, though that is a **convention** that
depends on the application.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
def iou(box_a, box_b):
    """2 つのボックスの IoU を計算する / compute IoU of two (x1, y1, x2, y2) boxes."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # 重なっている部分 / the intersection rectangle
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - intersection

    return intersection / union if union > 0 else 0.0


truth = (100, 100, 200, 200)          # 100 x 100 = 10,000 px2
cases = [
    ("完全一致 / identical      ", (100, 100, 200, 200)),
    ("少しずれる / shifted      ", (110, 110, 210, 210)),
    ("半分ずれる / half overlap ", (150, 100, 250, 200)),
    ("まったく別 / disjoint     ", (300, 300, 400, 400)),
]

print("正解 / truth :", truth)
print()
for label, box in cases:
    print(f"{label}  {box}  ->  IoU = {iou(truth, box):.3f}")

### 実行結果の例 / Expected output

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
正解 / truth : (100, 100, 200, 200)

完全一致 / identical        (100, 100, 200, 200)  ->  IoU = 1.000
少しずれる / shifted        (110, 110, 210, 210)  ->  IoU = 0.681
半分ずれる / half overlap   (150, 100, 250, 200)  ->  IoU = 0.333
まったく別 / disjoint       (300, 300, 400, 400)  ->  IoU = 0.000
```

This example produces the following result (the same input gives the same result).

```text
正解 / truth : (100, 100, 200, 200)

完全一致 / identical        (100, 100, 200, 200)  ->  IoU = 1.000
少しずれる / shifted        (110, 110, 210, 210)  ->  IoU = 0.681
半分ずれる / half overlap   (150, 100, 250, 200)  ->  IoU = 0.333
まったく別 / disjoint       (300, 300, 400, 400)  ->  IoU = 0.000
```


#### 手計算で確認する
「少しずれる」場合を手で計算してみましょう。

- 正解：`(100,100)` 〜 `(200,200)` → 面積 **10,000**
- 予測：`(110,110)` 〜 `(210,210)` → 面積 **10,000**
- 重なり：`(110,110)` 〜 `(200,200)` → 90 × 90 = **8,100**
- 2 つの枠を合わせた領域（和集合）：10,000 + 10,000 − 8,100 = **11,900**
- IoU = 8,100 ÷ 11,900 ≒ **0.681**

#### Verify by hand
Let us compute the "shifted" case by hand.

- truth: `(100,100)` to `(200,200)` -> area **10,000**
- prediction: `(110,110)` to `(210,210)` -> area **10,000**
- overlap: `(110,110)` to `(200,200)` -> 90 x 90 = **8,100**
- union of the two boxes: 10,000 + 10,000 - 8,100 = **11,900**
- IoU = 8,100 / 11,900 = **0.681**


---

#### 四角を動かして、交差・和集合・IoU を見る
#### Move a box and watch the intersection, the union and IoU

`offset_x` と `offset_y` だけを動かします。**予測の四角** が動きます。

Only `offset_x` and `offset_y` change — the **predicted** box moves.

| 予想 / Predict | 見るもの / What to look for |
|---|---|
| ずれを 0 にする / offset 0 | IoU はいくつ？ / what is IoU? |
| ずれを大きくする / increase the offset | 交差と和集合はどうなる？ / what happens to intersection and union? |
| まったく重ならない / no overlap | IoU はいくつ？ / what is IoU? |

ウィジェットが無くても、下のパラメータセルで同じ実験ができます。
The parameter cell below runs the same experiment without a widget.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import matplotlib.pyplot as plt

truth_box = (60, 60, 110, 90)     # (x, y, 幅, 高さ) / (x, y, width, height)
offset_x = 40                     # ← 動かす値 / change this
offset_y = 20                     # ← 動かす値 / change this
predicted_box = (truth_box[0] + offset_x, truth_box[1] + offset_y, truth_box[2], truth_box[3])


def intersection_and_union(box_a, box_b):
    left = max(box_a[0], box_b[0])
    top = max(box_a[1], box_b[1])
    right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
    bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
    intersection = max(0, right - left) * max(0, bottom - top)
    union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
    return intersection, union


intersection, union = intersection_and_union(truth_box, predicted_box)
print(f"交差 / intersection = {intersection} px²")
print(f"和集合 / union      = {union} px²")
print(f"IoU                 = {iou(truth_box, predicted_box):.3f}")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
ax.add_patch(plt.Rectangle(truth_box[:2], truth_box[2], truth_box[3], fill=False,
                           edgecolor="#1f6feb", linewidth=2))
ax.add_patch(plt.Rectangle(predicted_box[:2], predicted_box[2], predicted_box[3], fill=False,
                           edgecolor="#9a6700", linewidth=2, linestyle="--"))
ax.set_title(f"IoU = {iou(truth_box, predicted_box):.3f}", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(offset_x, offset_y):
        box_a = (60, 60, 110, 90)
        box_b = (box_a[0] + offset_x, box_a[1] + offset_y, box_a[2], box_a[3])
        left, top = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
        right = min(box_a[0] + box_a[2], box_b[0] + box_b[2])
        bottom = min(box_a[1] + box_a[3], box_b[1] + box_b[3])
        intersection = max(0, right - left) * max(0, bottom - top)
        union = box_a[2] * box_a[3] + box_b[2] * box_b[3] - intersection
        fig, ax = plt.subplots(figsize=(5.2, 5.2))
        ax.set_xlim(0, 300); ax.set_ylim(260, 0); ax.grid(alpha=0.2)
        ax.add_patch(plt.Rectangle(box_a[:2], box_a[2], box_a[3], fill=False,
                                   edgecolor="#1f6feb", linewidth=2))
        ax.add_patch(plt.Rectangle(box_b[:2], box_b[2], box_b[3], fill=False,
                                   edgecolor="#9a6700", linewidth=2, linestyle="--"))
        ax.set_title(f"intersection={intersection}  union={union}  "
                     f"IoU={iou(box_a, box_b):.3f}", fontsize=10)
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    widgets.interact(draw,
                     offset_x=widgets.IntSlider(value=40, min=-100, max=140, step=5,
                                                description="offset x"),
                     offset_y=widgets.IntSlider(value=20, min=-100, max=140, step=5,
                                                description="offset y"))
    display(output)

## Part 4　事前学習モデルと重みファイル / Pretrained models and weight files

#### 「学習済み」を持ってくる
ゼロから学習すると、大量の画像と長い計算時間が必要になります。
どれだけ必要かは、モデル・データ・計算機で大きく変わります。
そこで、**すでに大量の画像で学習済みのモデル** を使います。これを
**事前学習モデル（pretrained model）** と呼びます。

**重みファイル（weight file）** は、その学習結果が入ったファイルです。

| ファイル | 中身 |
|---|---|
| `.pt`（PyTorch） | **学習のチェックポイント**。モデルの構成情報、学習で決まったパラメータ、学習途中の状態が入っています |
| `.onnx`（ONNX） | **書き出した計算グラフとパラメータ**。学習の状態ではなく、別の実行環境で動かすための形式です |

`.pt` を `.onnx` に書き出すと、学習の記録（エポック数や最適化の状態）は入りません。
「学習を続けられるファイル」と「別の環境で実行するファイル」は別物です。

| 用語 | 意味 |
|---|---|
| 事前学習モデル | 他のデータで学習済みのモデル |
| 重み（weights） | 学習で決まった数値の集まり |
| 推論（inference） | 学習済みモデルで新しい画像を判定すること |
| ファインチューニング | 事前学習モデルを、自分のデータで追加学習すること |

第 4 日で行うのは **ファインチューニング** です。
COCO で学んだ「物の形を見る力」を借りて、PVC の欠陥を覚えさせます。
この研修の記録では、第 4 日のファインチューニング（15 エポック、CPU）は約 90 秒で終わりました
（`training_results/day04/pvc_defect_quick/results.csv` の `time` 列、最終行 89.7 秒）。
研修のデータセットが小さいためです。画像やエポックを増やすと、時間も増えます。

#### Starting from something already trained
Training from scratch needs a large image set and a long computation; how much depends on the
model, the data and the hardware. Instead we start from a model **already trained on a large
dataset**: a **pretrained model**.

A **weight file** holds the result of that training.

| File | Contents |
|---|---|
| `.pt` (PyTorch) | a **training checkpoint**: model configuration, the parameters fixed by training, and training state |
| `.onnx` (ONNX) | an **exported computation graph with its parameters**: a format for running the model in another environment, not a training state |

Exporting a `.pt` model to `.onnx` does not carry the training record (epoch count, optimiser
state). A file you can keep training and a file you can run elsewhere are different things.

| Term | Meaning |
|---|---|
| pretrained model | a model already trained on other data |
| weights | the numbers determined by training |
| inference | applying a trained model to new images |
| fine-tuning | further training of a pretrained model on your own data |

Day 4 performs **fine-tuning**: borrow the "shape understanding" learned on COCO and teach it
PVC defects. In this course's recorded run, fine-tuning (15 epochs, CPU) took about 90 seconds
(the `time` column of `training_results/day04/pvc_defect_quick/results.csv`, final row 89.7 s)
because the course dataset is small. More images or epochs mean more time.


#### なぜ COCO 事前学習モデルは PVC 欠陥を検出できないか
理由は 4 つあります。

1. **クラスが無い（クラス定義が違う）** — COCO の 80 種類に `black_spot` は無い。
   モデルは「黒い点」を何かに分類しようとするが、正解になるクラスが無い。
2. **写している分野（ドメイン）が違う** — COCO は人・車・動物などの日常の写真です。
   ただし COCO にも小さい物体や混み合った物体は含まれています。違うのは
   **何を写した画像か** と、**何を 1 つの物体と数えるか** という定義です。
3. **大きさと撮影条件が違う** — シート面の 0.3 mm の欠陥は、COCO の写真の物体と比べて
   極端に小さく、照明・距離・背景・材質も違います。
4. **正解データが無い** — そもそも「これは欠陥」というラベルで学習していない。

だから第 4 日で、**自分のデータで学習（ファインチューニング）** します。
第 3 日は「道具の使い方」を覚える日だと考えてください。

#### Why a COCO-pretrained model cannot detect PVC defects
Four reasons:

1. **no such class (a different class definition)** — `black_spot` is not among the 80 COCO
   classes, so there is no correct class for the model to predict
2. **a different domain** — COCO holds everyday photographs of people, vehicles and animals.
   COCO also contains small and crowded objects. What differs is **what the images show** and
   **what counts as one object**.
3. **different scale and imaging conditions** — a 0.3 mm mark on a sheet is far smaller than the
   objects in COCO photographs, and the lighting, distance, background and material differ
4. **no supervision** — the model was never trained with "this is a defect" labels

That is why Day 4 **fine-tunes on your own data**. Day 3 is about learning to use the tool.


---

## 演習 / Exercises

#### 演習 1（基本）：YOLO 形式に変換する
画像サイズが 640 × 480、ボックスが `x1=320, y1=200, x2=400, y2=260` のとき、
YOLO 形式の 4 つの値を計算してください。

完了の目安：4 つの値がすべて 0〜1 の範囲にあり、小数 6 桁で表示されていること。

#### Exercise 1 (basic): convert to YOLO format
For a 640x480 image and a box `x1=320, y1=200, x2=400, y2=260`, compute the four
YOLO values.
Done when all four values are between 0 and 1 and printed with 6 decimals.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
width, height = 640, 480
x1, y1, x2, y2 = 320, 200, 400, 260

xc = 0.0   # <- 計算 / compute
yc = 0.0
w = 0.0
h = 0.0

print(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

#### 演習 2（中級）：IoU を手計算して確かめる
正解 `(50, 50, 150, 150)`、予測 `(100, 50, 200, 150)` の IoU を
**紙で計算** してから、下のセルで確認してください。
重なりの面積が、2 つの枠を合わせた領域の何割になるかを考えます。

完了の目安：紙で計算した値と `iou(...)` の値が一致すること（0.333）。

#### Exercise 2 (intermediate): a hand calculation of IoU
Compute the IoU of truth `(50, 50, 150, 150)` and prediction `(100, 50, 200, 150)`
**on paper**, then check it with the cell. Think about what share of the union of the two boxes
the intersection covers.
Done when your paper result matches `iou(...)` (it is 0.333).


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
print("紙の計算 / on paper:", "")
print("コードの値 / code   :", round(iou((50, 50, 150, 150), (100, 50, 200, 150)), 3))

#### 演習 3（発展・任意）：第4日で何を学ぶ必要があるかを整理する
事前学習モデルが PVC 欠陥を検出できない 4 つの理由を踏まえ、
第 4 日で用意すべきものを整理してください。

| 項目 | あなたの答え |
|---|---|
| クラス名 | |
| 1 枚あたりの欠陥数（目安） | |
| 必要な画像枚数（学習用） | |
| 正解データの形式 | |

完了の目安：表が埋まり、ラベル形式の欄が YOLO の中心形式（`class_id x_center y_center width height`）になっていること。

#### Exercise 3 (optional challenge): plan what Day 4 must teach the model
Given the four reasons, plan what Day 4 must prepare: class names, defects per image,
number of training images, and the ground-truth format.
Done when the table is filled and the label-format field states the YOLO centre format
(`class_id x_center y_center width height`).


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
plan = {
    "classes": [],
    "defects_per_image": "",
    "training_images": "",
    "label_format": "",
}
for key, value in plan.items():
    print(f"{key:<20} {value if value else '(未記入 / not written)'}")

---

#### 理解度チェック / Understanding questions

1. YOLO の出力は何ですか。3 つの要素を挙げてください。
2. confidence が 0.9 の検出は、必ず正しいですか。理由も書いてください。
3. IoU が 0.3 のとき、重なりの面積は「何の」30 % ですか。
4. 事前学習モデルが PVC 欠陥を検出できない理由を 2 つ挙げてください。
5. 「推論」と「学習」の違いを、時間の観点から説明してください。

1. What does YOLO output? Name three components.
2. Is a detection with confidence 0.9 always correct? Explain.
3. When IoU is 0.3, the intersection is 30% of what?
4. Give two reasons a pretrained model cannot detect PVC defects.
5. Explain the difference between inference and training in terms of time.
Model answers: `docs/instructor/day03_instructor_guide.md`.


---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| YOLO 形式の値が 1 を超える | YOLO values exceed 1 | 画像の幅・高さで割り忘れています。`/ image_width` を確認してください。 | You forgot to divide by the image size. |
| IoU がマイナスになる | IoU is negative | 重なりが無いときは 0 になります。`max(0.0, ...)` を忘れていないか確認してください。 | It should be 0 when there is no overlap; check `max(0.0, ...)`. |
| `confidence` と `確率` を混同してしまう | Confusing confidence with probability | confidence はモデルが出したスコアです。正しい確率ではありません。正しさは正解ラベルと比べて確認します。 | Confidence is a score produced by the model, not a probability of correctness; check correctness against ground truth. |
| 数値がノートの例と少し違う | Numbers differ slightly | 丸めの桁数が違うだけのことが多いです。`round()` の桁を確認してください。 | Usually just a rounding difference; check the digits in `round()`. |


## 完了チェックリスト / Completion checklist

**必須**は下の「必須 / Required」だけです。**演習 3 と追加チャレンジは任意（発展・任意）** で、
未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the items under **必須 / Required** are mandatory. **Exercise 3 and the extra challenge are
optional**; leaving them undone still counts as complete. Do them only if you have spare time.

**必須 / Required**

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] YOLO 形式の座標を自分で計算できた
      - [ ] Computed YOLO coordinates yourself
- [ ] IoU を手計算し、コードと一致した
      - [ ] Hand-computed IoU and matched the code
- [ ] 演習 1・演習 2（必須）を実施した
      - [ ] Completed Exercises 1 and 2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 understanding questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

**任意 / Optional**

- [ ] 演習 3（発展・任意）：第 4 日の準備項目を整理した
      - [ ] Exercise 3 (optional challenge): planned what Day 4 must prepare
- [ ] 追加チャレンジ：COCO の 80 クラスから製造検査に使えそうなものを探した
      - [ ] Extra challenge: looked for COCO classes useful for inspection


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics ドキュメント](https://docs.ultralytics.com/)
- [Ultralytics: 物体検出タスク](https://docs.ultralytics.com/tasks/detect/)
- [COCO データセットのクラス一覧](https://cocodataset.org/#explore)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

COCO の 80 クラスの一覧を見て、製造検査に使えそうなクラスがあるか探してください。

Look at the 80 COCO classes and see whether any of them could be useful for inspection.